# Parse Golden LLM Answers

Ce notebook importe `swissdox_prompts_golden.csv`, décompose la colonne `llm_answer` en colonnes structurées,
exporte `swissdox_full_article_gold.csv` et un fichier de synthèse textuelle.

In [1]:
import csv
import re
import pandas as pd
from pathlib import Path

csv.field_size_limit(10**7)

DATA_DIR = Path('../data/external')
INPUT_FILE  = DATA_DIR / 'swissdox_prompts_golden.csv'
OUTPUT_CSV  = DATA_DIR / 'swissdox_full_article_gold.csv'
OUTPUT_TXT  = DATA_DIR / 'swissdox_full_article_gold_summary.txt'

FIELDS = [
    'SWISS_CONTEXT',
    'CRITICISM',
    'TARGETED_ENTITY_TYPE',
    'TARGETED_ENTITY_NAME',
    'SOURCE_TYPE',
    'SOURCE_NAME',
    'CRITICISM_TOPIC',
    'POPULIST_RHETORIC',
]

print(f'Input : {INPUT_FILE}')
print(f'Output CSV : {OUTPUT_CSV}')
print(f'Output TXT : {OUTPUT_TXT}')

Input : ../data/external/swissdox_prompts_golden.csv
Output CSV : ../data/external/swissdox_full_article_gold.csv
Output TXT : ../data/external/swissdox_full_article_gold_summary.txt


In [2]:
df = pd.read_csv(INPUT_FILE, dtype=str)
print(f'Rows loaded : {len(df)}')
print(f'Columns     : {list(df.columns)}')

Rows loaded : 8787
Columns     : ['article_id', 'title', 'lead', 'pubtime', 'medium_code', 'medium_name', 'rubric', 'regional', 'doctype', 'doctype_description', 'language', 'char_count', 'dateline', 'prompt_ready', 'llm_answer']


## Parsing de la colonne `llm_answer`

Le format attendu est une séquence de paires `FIELD: value` sur une seule ligne.
On utilise un regex qui identifie chaque champ comme délimiteur pour extraire les valeurs,
y compris les valeurs multi-mots (comme CRITICISM_TOPIC).

In [3]:
# Build a regex that splits on known field names used as delimiters
# Pattern: capture everything between two consecutive field tags
_FIELD_PATTERN = re.compile(
    r'(?:' + '|'.join(re.escape(f) for f in FIELDS) + r'):\s*'
)

def parse_llm_answer(text: str) -> dict:
    """Parse a single llm_answer string into a dict of {field: value}."""
    result = {f: None for f in FIELDS}
    if not isinstance(text, str) or not text.strip():
        return result

    # Find all field positions
    # Build a combined pattern to find (field_name, start_of_value) pairs
    tag_re = re.compile(
        r'(' + '|'.join(re.escape(f) for f in FIELDS) + r'):\s*'
    )
    matches = list(tag_re.finditer(text))

    for i, m in enumerate(matches):
        field_name = m.group(1)
        value_start = m.end()
        # Value ends at the start of the next field tag (or end of string)
        value_end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        value = text[value_start:value_end].strip()
        result[field_name] = value if value else None

    return result

# Quick test
sample = (
    "SWISS_CONTEXT: YES CRITICISM: YES "
    "TARGETED_ENTITY_TYPE: Federal Department "
    "TARGETED_ENTITY_NAME: le Département fédéral "
    "SOURCE_TYPE: Another Federal Department "
    "SOURCE_NAME: le département de l'Économie "
    "CRITICISM_TOPIC: The Department criticises the proposal. "
    "POPULIST_RHETORIC: NO"
)
parse_llm_answer(sample)

{'SWISS_CONTEXT': 'YES',
 'CRITICISM': 'YES',
 'TARGETED_ENTITY_TYPE': 'Federal Department',
 'TARGETED_ENTITY_NAME': 'le Département fédéral',
 'SOURCE_TYPE': 'Another Federal Department',
 'SOURCE_NAME': "le département de l'Économie",
 'CRITICISM_TOPIC': 'The Department criticises the proposal.',
 'POPULIST_RHETORIC': 'NO'}

In [4]:
# Apply parsing to all rows
parsed = df['llm_answer'].apply(parse_llm_answer)
parsed_df = pd.DataFrame(parsed.tolist())

# Count rows where no field was extracted (parsing failed)
n_failed = (parsed_df[FIELDS[0]].isna() & df['llm_answer'].notna()).sum()
print(f'Rows with parsing failure : {n_failed}')
print(f'Parsed columns : {list(parsed_df.columns)}')
parsed_df.head(3)

Rows with parsing failure : 0
Parsed columns : ['SWISS_CONTEXT', 'CRITICISM', 'TARGETED_ENTITY_TYPE', 'TARGETED_ENTITY_NAME', 'SOURCE_TYPE', 'SOURCE_NAME', 'CRITICISM_TOPIC', 'POPULIST_RHETORIC']


,SWISS_CONTEXT,CRITICISM,TARGETED_ENTITY_TYPE,TARGETED_ENTITY_NAME,SOURCE_TYPE,SOURCE_NAME,CRITICISM_TOPIC,POPULIST_RHETORIC
0,YES,NO,N/A,N/A,N/A,N/A,N/A,N/A
1,YES,YES,Federal Department,"le Département fédéral de l’environnement, des...",Another Federal Department,le département de l’Économie,The Department of the Economy criticises the p...,NO
2,YES,YES,Federal Department,Aussendepartement (EDA),Journalist (author or editorial stance),Benedikt Hollenstein,The article criticises the EDA for keeping a l...,NO


In [5]:
# Merge original dataframe with parsed columns
df_full = pd.concat([df, parsed_df], axis=1)
print(f'Final shape : {df_full.shape}')
df_full.head(3)

Final shape : (8787, 23)


,article_id,title,lead,pubtime,medium_code,medium_name,rubric,regional,doctype,doctype_description,...,prompt_ready,llm_answer,SWISS_CONTEXT,CRITICISM,TARGETED_ENTITY_TYPE,TARGETED_ENTITY_NAME,SOURCE_TYPE,SOURCE_NAME,CRITICISM_TOPIC,POPULIST_RHETORIC
0,0001de76-bb46-1fd0-ff60-980d765e15a8,Mega-Angriff legt Schweizer Seiten lahm: Hacke...,NoName057(16) bekennt sich zu DDoS-Angriffen a...,2025-01-21,ZWAO,20 minuten online,NaN,NaN,WWE,Online medium,...,You are a strict media analysis assistant spec...,SWISS_CONTEXT: YES CRITICISM: NO TARGETED_ENTI...,YES,NO,N/A,N/A,N/A,N/A,N/A,N/A
1,000658f2-9b7b-c9b4-9592-008f8f4d0b54,Vers la fin des vols en classe affaires pour l...,Privilèges Albert Rösti propose de réviser les...,2025-09-30,HEU,24 heures,Actualités,NaN,PRD,Regional daily newspaper,...,You are a strict media analysis assistant spec...,SWISS_CONTEXT: YES CRITICISM: YES TARGETED_ENT...,YES,YES,Federal Department,"le Département fédéral de l’environnement, des...",Another Federal Department,le département de l’Économie,The Department of the Economy criticises the p...,NO
2,0009a49c-b08a-7e36-0544-bdbeaec73b95,EDA hält Palästina-Infos zurück – «zum Schutz ...,Ein EDA-Gutachten zur Anerkennung Palästinas b...,2025-09-21,ZWAO,20 minuten online,NaN,NaN,WWE,Online medium,...,You are a strict media analysis assistant spec...,SWISS_CONTEXT: YES CRITICISM: YES TARGETED_ENT...,YES,YES,Federal Department,Aussendepartement (EDA),Journalist (author or editorial stance),Benedikt Hollenstein,The article criticises the EDA for keeping a l...,NO


In [6]:
# Export CSV
df_full.to_csv(OUTPUT_CSV, index=False)
print(f'CSV exporté : {OUTPUT_CSV}')

CSV exporté : ../data/external/swissdox_full_article_gold.csv


## Synthèse des résultats

In [7]:
# Columns to summarise (all except CRITICISM_TOPIC)
SUMMARY_FIELDS = [f for f in FIELDS if f != 'CRITICISM_TOPIC']

total = len(df_full)
lines = []
lines.append('=' * 70)
lines.append('SYNTHÈSE DES RÉSULTATS — swissdox_full_article_gold')
lines.append(f'Total articles analysés : {total}')
lines.append('=' * 70)

for field in SUMMARY_FIELDS:
    lines.append('')
    lines.append(f'── {field} ──')
    counts = df_full[field].fillna('(manquant)').value_counts()
    for val, n in counts.items():
        pct = 100 * n / total
        lines.append(f'  {val:<45} {n:>6}  ({pct:5.1f}%)')

lines.append('')
lines.append('=' * 70)
summary_text = '\n'.join(lines)
print(summary_text)

SYNTHÈSE DES RÉSULTATS — swissdox_full_article_gold
Total articles analysés : 8787

── SWISS_CONTEXT ──
  (manquant)                                      8588  ( 97.7%)
  YES                                              196  (  2.2%)
  NO                                                 3  (  0.0%)

── CRITICISM ──
  (manquant)                                      8588  ( 97.7%)
  YES                                              109  (  1.2%)
  NO                                                87  (  1.0%)
  N/A                                                3  (  0.0%)

── TARGETED_ENTITY_TYPE ──
  (manquant)                                      8588  ( 97.7%)
  N/A                                               90  (  1.0%)
  Other                                             28  (  0.3%)
  Federal Agency or Civil Servant                   20  (  0.2%)
  Federal Department                                19  (  0.2%)
  Political Figure                                  16  (  0.2%)
  Muni

In [8]:
# Export TXT
OUTPUT_TXT.write_text(summary_text, encoding='utf-8')
print(f'Synthèse exportée : {OUTPUT_TXT}')

Synthèse exportée : ../data/external/swissdox_full_article_gold_summary.txt
